# Round 2 — 모자 데이터 추가 fine-tuning
- 베이스 모델: `runs2/round1/weights/best.pt`
- 데이터: 모자 데이터 추가된 yaml
- 저장 위치: `runs2/round2`
- MLflow: 자동 로깅

In [ ]:
import mlflow
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────
# round2용 데이터셋 yaml (모자 데이터 추가된 버전)
DATA_YAML   = '/mnt/ml/helmet_project/data/azure_blob/data_hat/data.yaml'
PROJECT_DIR = '/mnt/ml/helmet_project/experiments/runs2'
RUN_NAME    = 'round2'
BASE_MODEL  = '/mnt/ml/helmet_project/experiments/runs2/round1/weights/best.pt'  # round1 이어받기

# ── MLflow 설정 ────────────────────────────────────
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('helmet_detection')

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        'base_model' : BASE_MODEL,
        'data'       : DATA_YAML,
        'epochs'     : 50,
        'imgsz'      : 640,
        'batch'      : 16,
        'round'      : 2,
        'cls_weight' : 2.0,   # no_helmet 클래스에 집중
        'note'       : 'hat_data_added_finetune',
    })

    model = YOLO(BASE_MODEL)  # round1 best.pt 로드

    results = model.train(
        data     = DATA_YAML,
        epochs   = 50,
        imgsz    = 640,
        batch    = 16,
        device   = 0,
        project  = PROJECT_DIR,
        name     = RUN_NAME,
        exist_ok = True,
        cls      = 2.0,        # no_helmet 오탐 줄이기
        lr0      = 0.001,      # fine-tuning이므로 lr 낮춤
        warmup_epochs = 3,
    )

    metrics = results.results_dict
    mlflow.log_metrics({
        'mAP50'     : metrics.get('metrics/mAP50(B)', 0),
        'mAP50_95'  : metrics.get('metrics/mAP50-95(B)', 0),
        'precision' : metrics.get('metrics/precision(B)', 0),
        'recall'    : metrics.get('metrics/recall(B)', 0),
    })

    best_pt = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
    mlflow.log_artifact(best_pt, artifact_path='weights')

    print(f'\n✅ Round2 완료')
    print(f'   best.pt: {best_pt}')
    print(f'   mAP50  : {metrics.get("metrics/mAP50(B)", 0):.4f}')

In [ ]:
# 검증 (선택)
from ultralytics import YOLO

model = YOLO('/mnt/ml/helmet_project/experiments/runs2/round2/weights/best.pt')
val = model.val(data='/mnt/ml/helmet_project/data/azure_blob/data_hat/data.yaml', split='val')
print(val)